## 8. 常见问题 (FAQ)

### Q1: 为什么叫"自"注意力？

**A**: 因为 Query、Key、Value 都来自同一个序列（自己），而不是像 Seq2Seq 中的注意力那样来自不同序列。

### Q2: 注意力权重的和一定是 1 吗？

**A**: 是的。Softmax 函数保证每一行的权重和为 1，这样可以理解为概率分布。

### Q3: 为什么要除以 $\sqrt{d_k}$？

**A**: 当 $d_k$ 很大时，点积的值会很大，导致 softmax 进入饱和区，梯度接近 0。缩放可以保持梯度稳定。

### Q4: 多头注意力的参数量会增加吗？

**A**: 不会。虽然有多个头，但每个头的维度 $d_k = d_{model}/h$ 更小，总参数量与单头相同。

### Q5: 如何选择头的数量？

**A**: 常见配置：
- 小模型：4 个头
- 基础模型：8 个头
- 大模型：16-32 个头

一般 $d_{model}$ 要能被 $h$ 整除。

### Q6: 自注意力的复杂度是多少？

**A**: 
- **时间复杂度**：$O(n^2 \cdot d)$，其中 $n$ 是序列长度，$d$ 是模型维度
- **空间复杂度**：$O(n^2)$，需要存储注意力矩阵

这是 Transformer 处理长序列的主要瓶颈。

### Q7: 如何处理变长序列？

**A**: 使用 padding 和 mask：
1. 将短序列 padding 到相同长度
2. 使用 mask 标记 padding 位置
3. 在计算注意力时，将 padding 位置的权重设为 0

### Q8: 注意力可以替代 RNN 吗？

**A**: 在大多数任务上可以，且效果更好：
- **优势**：并行计算、长程依赖、可解释性
- **劣势**：序列很长时内存占用大

### Q9: 如何理解不同头学到的模式？

**A**: 通过可视化注意力权重：
- 对角线模式：关注相邻位置
- 垂直/水平线：关注特定位置
- 分散模式：关注多个相关位置

### Q10: 自注意力有位置信息吗？

**A**: 没有！自注意力本身是位置不变的（permutation invariant）。需要通过位置编码（Positional Encoding）注入位置信息，这将在下一章学习。

---

## 🎓 思考题

1. 如果不使用缩放，会发生什么？
2. 单头注意力和多头注意力的表达能力有什么区别？
3. 如何修改注意力机制来处理超长序列？
4. 注意力权重可以用于模型解释吗？有什么局限性？

---

**恭喜完成自注意力机制的学习！** 🎉

你已经掌握了 Transformer 的核心组件。继续学习下一章，构建完整的 Transformer 编码器！

## 思考题参考答案

### 问题 1：如果不使用缩放，会发生什么？

**答案：**

不使用缩放因子 $\frac{1}{\sqrt{d_k}}$ 时，注意力计算变为：

$$\text{Attention}(Q, K, V) = \text{softmax}(QK^T)V$$

**具体会发生的问题：**

1. **点积数值过大**：当 $d_k$ 较大时（如 64），$Q$ 和 $K$ 各维度通常满足均值为 0、方差为 1 的分布，则点积 $QK^T$ 的方差约为 $d_k$，标准差约为 $\sqrt{d_k}$。对于 $d_k = 64$，标准差约为 8，数值量级较大。

2. **Softmax 饱和（梯度消失）**：点积过大导致 softmax 函数进入饱和区。例如对向量 $[100, 1, 2]$ 做 softmax，结果近似为 $[1.0, 0.0, 0.0]$，梯度接近于零，模型无法有效学习。

3. **注意力分布退化**：模型趋向于将几乎所有注意力集中在一个位置，失去多样化关注的能力。

**缩放后的效果：**

$$\text{Var}\left(\frac{QK^T}{\sqrt{d_k}}\right) = 1$$

方差归一化到 1，softmax 梯度处于合理范围，训练稳定。

**数值示例对比：**

| 场景 | 输入值 | Softmax 输出 | 梯度量级 |
|------|--------|-------------|---------|
| 无缩放 ($d_k=64$) | $[8, -8, 0]$ | $\approx [1.0, 0.0, 0.0]$ | 极小 |
| 有缩放 ($d_k=64$) | $[1, -1, 0]$ | $\approx [0.58, 0.21, 0.21]$ | 正常 |

---

### 问题 2：单头注意力和多头注意力的表达能力有什么区别？

**答案：**

**单头注意力的局限性：**

单头注意力只能学习一种注意力模式，在每个时刻只能关注序列中的一种关系（如局部相邻关系）。它的表达是：

$$\text{head} = \text{Attention}(QW^Q, KW^K, VW^V)$$

对于复杂的语言现象（如同时存在语法关系和语义关系），单头难以同时捕获。

**多头注意力的优势：**

多头注意力在不同的投影子空间中并行计算注意力：

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

不同的头可以专门化地学习不同模式，例如：

- **Head 1**：句法关系（主语-谓语-宾语）
- **Head 2**：指代关系（代词 → 先行词）
- **Head 3**：局部位置关系（相邻词）
- **Head 4**：长程语义依赖

**参数量对比：**

| 类型 | 参数量 | 子空间维度 | 表达能力 |
|------|--------|-----------|---------|
| 单头 | $4 \times d_{model}^2$ | $d_{model}$ | 单一模式 |
| 多头 ($h$ 头) | $4 \times d_{model}^2$ | $d_k = d_{model}/h$ | $h$ 种模式并行 |

**关键结论：** 多头注意力在不增加参数量的情况下，通过并行子空间显著增强了模型的表达能力。每个头学习的是 $d_k$ 维度的特征空间，而非完整的 $d_{model}$ 维度，这使得各头可以专注于不同的语言现象。

---

### 问题 3：如何修改注意力机制来处理超长序列？

**答案：**

标准自注意力的时间和空间复杂度均为 $O(n^2)$，对于超长序列（如 $n > 4096$）代价过高。以下是主流改进方案：

**方案一：稀疏注意力（Sparse Attention）**

限制每个 token 只关注序列中的一个子集：

- **局部窗口注意力**：每个 token 只关注其左右 $w$ 个邻居，复杂度 $O(n \cdot w)$
- **扩张注意力（Dilated Attention）**：以步长 $d$ 采样，扩大感受野
- **全局 token + 局部注意力**：少数全局 token 可以关注所有位置（如 Longformer）

**方案二：线性注意力（Linear Attention）**

用核函数近似 softmax，将复杂度降至 $O(n)$：

$$\text{Attention}(Q, K, V) \approx \phi(Q)\left(\phi(K)^T V\right)$$

利用结合律，先计算 $\phi(K)^T V$（$d \times d$ 矩阵），再乘以 $\phi(Q)$，复杂度为 $O(n \cdot d^2)$，代表方法：Performer、Linear Transformer。

**方案三：分块注意力**

将序列分成若干块，块内做完整注意力，块间做稀疏注意力（如 BigBird）。

**方案四：滑动窗口 + 全局 Token（Longformer）**

```
局部：每个 token 关注 ±w 窗口内的 token
全局：特殊 token（如 [CLS]）关注全部序列
```

**方案五：Flash Attention（IO 优化）**

不减少浮点运算量，而是通过分块计算降低 HBM 访问次数，在保持精确计算的同时显著加速，目前已成为工程实践中的标配。

**方案对比：**

| 方法 | 复杂度 | 是否精确 | 典型代表 |
|------|--------|---------|---------|
| 稀疏注意力 | $O(n \sqrt{n})$ 或更低 | 近似 | Longformer, BigBird |
| 线性注意力 | $O(n)$ | 近似 | Performer |
| Flash Attention | $O(n^2)$（但内存高效） | 精确 | FlashAttention v1/v2/v3 |

---

### 问题 4：注意力权重可以用于模型解释吗？有什么局限性？

**答案：**

**可以使用的方面：**

注意力权重直观地显示了模型在生成输出时"关注"了哪些输入 token，被广泛用于：

- 可视化词语对齐关系（机器翻译任务）
- 分析代词指代关系（如 "it" 指代 "cat" 还是 "mat"）
- 理解模型对关键词的关注程度（情感分析）

**主要局限性：**

**1. 注意力 ≠ 重要性（Attention is not Explanation）**

Jain & Wallace（2019）和 Wiegreffe & Pinter（2019）的研究表明，注意力权重与梯度定义的特征重要性（如 LIME、SHAP）往往不一致。可以构造出权重不同但输出相同的注意力分布。

**2. 多头的聚合问题**

多头注意力有多个头，每个头的权重模式各不相同，如何聚合成单一的解释并没有公认的方法。简单取平均会丢失信息。

**3. 残差连接的影响**

每一层的输出是 $\text{LayerNorm}(x + \text{Attention}(x))$，残差路径携带了大量信息，仅看注意力权重无法反映残差中直接传递的信息。

**4. 多层叠加的复合效应**

深层 Transformer 中，信息经过多层注意力叠加，底层注意力权重与最终输出之间的因果关系难以追踪。

**5. 头的异质性**

不同头学习到的模式功能各异，部分头可能是"垃圾回收"头（处理无关信息），直接解读其权重意义不大。

**更可靠的解释性方法：**

- **梯度 × 输入（Gradient × Input）**：将梯度与输入相乘来量化贡献
- **集成梯度（Integrated Gradients）**：沿路径积分梯度，满足完整性公理
- **SHAP / LIME**：模型无关的局部解释方法
- **注意力流（Attention Flow）**：跨层追踪信息流动

**结论：** 注意力权重可作为探索性分析工具，提供直觉线索，但不能作为严格的因果解释证据。


## 7. 本章总结 (Summary)

### 7.1 核心要点回顾

**自注意力机制**：
- Query、Key、Value 三个核心概念
- 通过点积计算相似度，softmax 归一化
- 缩放因子 $\sqrt{d_k}$ 保持梯度稳定

**多头注意力**：
- 并行运行多个注意力头
- 不同的头学习不同的模式
- 增强模型的表达能力

**关键优势**：
- 可以并行计算，训练效率高
- 直接建模长程依赖
- 注意力权重可解释

### 7.2 与 RNN 的对比

| 维度 | RNN | Self-Attention |
|------|-----|----------------|
| **并行性** | 顺序处理 | 完全并行 |
| **长程依赖** | 梯度消失 | 直接连接 |
| **计算复杂度** | O(n) | O(n²) |
| **内存复杂度** | O(1) | O(n²) |
| **可解释性** | 较差 | 较好 |

### 7.3 下一步学习

完成本章后，你应该：
- ✅ 理解自注意力的工作原理
- ✅ 能够实现缩放点积注意力
- ✅ 能够实现多头注意力
- ✅ 能够可视化注意力模式

**继续学习**：
- 下一章：Transformer 编码器（位置编码、前馈网络、层归一化）
- 扩展阅读：Attention Is All You Need 论文

In [ ]:
# 🔬 Micro Practice: Sentence Encoding with Self-Attention
# Goal: Use self-attention to encode a sentence
# Expected outcome: Rich sentence representation

class SelfAttentionEncoder(nn.Module):
    """Simple encoder using self-attention"""
    
    def __init__(self, vocab_size, d_model, num_heads):
        super(SelfAttentionEncoder, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.layer_norm = nn.LayerNorm(d_model)
    
    def forward(self, x):
        """
        Args:
            x: Token IDs (batch, seq_len)
        
        Returns:
            output: Encoded representation (batch, seq_len, d_model)
            attention_weights: Attention weights
        """
        # Embed tokens
        embedded = self.embedding(x)
        
        # Self-attention
        attn_output, attn_weights = self.attention(embedded, embedded, embedded)
        
        # Residual connection and layer norm
        output = self.layer_norm(embedded + attn_output)
        
        return output, attn_weights

# Example: Encode a sentence
print("=" * 60)
print("句子编码示例")
print("=" * 60)

# Simple vocabulary
vocab = {
    "<PAD>": 0, "The": 1, "cat": 2, "sat": 3, "on": 4, 
    "the": 5, "mat": 6, "dog": 7, "ran": 8
}
vocab_size = len(vocab)

# Create encoder
d_model = 128
num_heads = 4
encoder = SelfAttentionEncoder(vocab_size, d_model, num_heads)

# Encode two sentences
sentence1 = ["The", "cat", "sat", "on", "the", "mat"]
sentence2 = ["The", "dog", "ran"]

# Convert to IDs and pad
def encode_sentence(sentence, vocab, max_len=10):
    ids = [vocab.get(word, 0) for word in sentence]
    ids += [0] * (max_len - len(ids))  # Pad
    return torch.LongTensor([ids])

sent1_ids = encode_sentence(sentence1, vocab)
sent2_ids = encode_sentence(sentence2, vocab)

# Encode
output1, attn1 = encoder(sent1_ids)
output2, attn2 = encoder(sent2_ids)

print(f"\n句子 1: {' '.join(sentence1)}")
print(f"  Token IDs: {sent1_ids[0][:len(sentence1)].tolist()}")
print(f"  编码形状: {output1.shape}")

print(f"\n句子 2: {' '.join(sentence2)}")
print(f"  Token IDs: {sent2_ids[0][:len(sentence2)].tolist()}")
print(f"  编码形状: {output2.shape}")

# Compute sentence similarity using mean pooling
def mean_pool(output, lengths):
    """Average pooling over sequence"""
    mask = torch.arange(output.size(1)) < lengths
    mask = mask.unsqueeze(0).unsqueeze(-1).float()
    return (output * mask).sum(1) / lengths.float().unsqueeze(-1)

sent1_repr = mean_pool(output1, torch.tensor([len(sentence1)]))
sent2_repr = mean_pool(output2, torch.tensor([len(sentence2)]))

# Cosine similarity
similarity = F.cosine_similarity(sent1_repr, sent2_repr)

print(f"\n句子表示:")
print(f"  句子 1 向量: {sent1_repr.shape}")
print(f"  句子 2 向量: {sent2_repr.shape}")
print(f"  余弦相似度: {similarity.item():.4f}")

print("\n✓ 句子编码完成！")
print("\n说明:")
print("  - 自注意力捕获句子内部的关系")
print("  - 每个词的表示融合了上下文信息")
print("  - 可以用于下游任务（分类、相似度等）")

## 6. 实际应用示例 (Practical Example)

### 6.1 文本相似度计算

使用自注意力计算句子内部的词语相似度。

In [ ]:
# 🔬 Micro Practice: Visualize Multi-Head Attention
# Goal: Visualize different attention patterns from different heads
# Expected outcome: Clear visualization of attention patterns

def visualize_attention_heads(attention_weights, tokens=None, num_heads_to_show=4):
    """
    Visualize attention patterns from multiple heads
    
    Args:
        attention_weights: (batch, num_heads, seq_len, seq_len)
        tokens: Optional list of token strings
        num_heads_to_show: Number of heads to visualize
    """
    batch_idx = 0
    num_heads = min(num_heads_to_show, attention_weights.size(1))
    seq_len = attention_weights.size(2)
    
    # Create token labels if not provided
    if tokens is None:
        tokens = [f"T{i}" for i in range(seq_len)]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    axes = axes.flatten()
    
    for head_idx in range(num_heads):
        ax = axes[head_idx]
        
        # Get attention weights for this head
        attn = attention_weights[batch_idx, head_idx].detach().numpy()
        
        # Plot heatmap
        im = ax.imshow(attn, cmap='viridis', aspect='auto')
        
        # Set labels
        ax.set_xticks(range(seq_len))
        ax.set_yticks(range(seq_len))
        ax.set_xticklabels(tokens, rotation=45, ha='right')
        ax.set_yticklabels(tokens)
        
        ax.set_xlabel('Key (attending to)')
        ax.set_ylabel('Query (attending from)')
        ax.set_title(f'Head {head_idx + 1}')
        
        # Add colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        
        # Add grid
        for i in range(seq_len + 1):
            ax.axhline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
            ax.axvline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Create example with meaningful tokens
print("=" * 60)
print("多头注意力可视化")
print("=" * 60)

# Example sentence
tokens = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(tokens)
d_model = 512
num_heads = 8

# Create multi-head attention
mha = MultiHeadAttention(d_model, num_heads)

# Create input (random embeddings for demonstration)
x = torch.randn(1, seq_len, d_model)

# Get attention weights
output, attn_weights = mha(x, x, x)

print(f"\n句子: {' '.join(tokens)}")
print(f"序列长度: {seq_len}")
print(f"注意力头数: {num_heads}")
print(f"\n可视化前 4 个注意力头的模式：\n")

# Visualize
visualize_attention_heads(attn_weights, tokens, num_heads_to_show=4)

print("\n观察：")
print("  - 不同的头展现不同的注意力模式")
print("  - 有些头关注局部（对角线）")
print("  - 有些头关注特定位置")
print("  - 这种多样性增强了模型的表达能力")

print("\n✓ 注意力可视化完成！")

## 5. 注意力可视化 (Attention Visualization)

### 5.1 可视化的重要性

可视化注意力权重可以帮助我们：
- 理解模型在关注什么
- 调试模型行为
- 解释模型决策
- 发现潜在问题

### 5.2 不同头的注意力模式

多头注意力的不同头会学习不同的模式：
- **位置模式**：关注相邻位置
- **语法模式**：关注语法关系（主谓宾）
- **语义模式**：关注语义相关的词
- **长程依赖**：关注距离较远的词

In [ ]:
# 🔬 Micro Practice: Implement Multi-Head Attention
# Goal: Build complete multi-head attention module
# Expected outcome: Working MHA with multiple parallel heads

class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention Module
    
    Args:
        d_model: Model dimension (e.g., 512)
        num_heads: Number of attention heads (e.g., 8)
        dropout: Dropout rate
    """
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Dimension per head
        
        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k)
        
        Args:
            x: (batch, seq_len, d_model)
        
        Returns:
            (batch, num_heads, seq_len, d_k)
        """
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine heads back to original shape
        
        Args:
            x: (batch, num_heads, seq_len, d_k)
        
        Returns:
            (batch, seq_len, d_model)
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, query, key, value, mask=None):
        """
        Forward pass
        
        Args:
            query: (batch, seq_len, d_model)
            key: (batch, seq_len, d_model)
            value: (batch, seq_len, d_model)
            mask: Optional mask
        
        Returns:
            output: (batch, seq_len, d_model)
            attention_weights: (batch, num_heads, seq_len, seq_len)
        """
        batch_size = query.size(0)
        
        # 1. Linear projections
        Q = self.W_q(query)  # (batch, seq_len, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Split into multiple heads
        Q = self.split_heads(Q)  # (batch, num_heads, seq_len, d_k)
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3. Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = torch.nan_to_num(attention_weights, 0.0)
        attention_weights = self.dropout(attention_weights)
        
        # 4. Apply attention to values
        context = torch.matmul(attention_weights, V)  # (batch, num_heads, seq_len, d_k)
        
        # 5. Combine heads
        context = self.combine_heads(context)  # (batch, seq_len, d_model)
        
        # 6. Final linear projection
        output = self.W_o(context)
        
        return output, attention_weights

# Test Multi-Head Attention
print("=" * 60)
print("多头注意力测试")
print("=" * 60)

d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model, num_heads)

# Create input
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass (self-attention: Q=K=V)
output, attn_weights = mha(x, x, x)

print(f"\n配置:")
print(f"  d_model: {d_model}")
print(f"  num_heads: {num_heads}")
print(f"  d_k (per head): {d_model // num_heads}")

print(f"\n形状:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {attn_weights.shape}")

print(f"\n参数量:")
total_params = sum(p.numel() for p in mha.parameters())
print(f"  Total: {total_params:,} parameters")

print("\n✓ 多头注意力实现成功！")

## 4. 多头注意力 (Multi-Head Attention)

### 4.1 为什么需要多头？

**单头注意力的局限**：
- 只能学习一种注意力模式
- 表达能力有限

**多头注意力的优势**：
- 不同的头可以关注不同的模式
- 类似于 CNN 中的多个卷积核
- 增强模型的表达能力

**例子**：在翻译 "The animal didn't cross the street because it was too tired" 时：
- Head 1：关注语法关系（主谓宾）
- Head 2：关注语义关系（代词指代）
- Head 3：关注位置关系（相邻词）

### 4.2 多头注意力原理

**核心思想**：并行运行多个注意力头，然后拼接结果。

**数学公式**：

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

其中每个头：

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**参数**：
- $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$：Query 投影矩阵
- $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$：Key 投影矩阵
- $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$：Value 投影矩阵
- $W^O \in \mathbb{R}^{hd_v \times d_{model}}$：输出投影矩阵

**维度关系**：
- $d_k = d_v = d_{model} / h$
- 总参数量与单头相同

### 4.3 计算流程

```
输入 (batch, seq_len, d_model)
    ↓
[线性投影] → Q, K, V (batch, seq_len, d_model)
    ↓
[分割成 h 个头] → (batch, h, seq_len, d_k)
    ↓
[并行计算注意力] → (batch, h, seq_len, d_k)
    ↓
[拼接] → (batch, seq_len, h * d_k)
    ↓
[输出投影] → (batch, seq_len, d_model)
```

In [ ]:
# 🔬 Micro Practice: Implement Scaled Dot-Product Attention
# Goal: Build the core attention mechanism
# Expected outcome: Working attention function

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled Dot-Product Attention
    
    Args:
        Q: Query matrix (batch, seq_len, d_k)
        K: Key matrix (batch, seq_len, d_k)
        V: Value matrix (batch, seq_len, d_v)
        mask: Optional mask (batch, seq_len, seq_len)
    
    Returns:
        output: Attention output (batch, seq_len, d_v)
        attention_weights: Attention weights (batch, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Step 1: Compute attention scores (Q @ K^T)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (optional)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Apply softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Handle NaN from softmax(-inf)
    attention_weights = torch.nan_to_num(attention_weights, 0.0)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test the attention function
print("=" * 60)
print("缩放点积注意力测试")
print("=" * 60)

batch_size = 2
seq_len = 5
d_k = 64
d_v = 64

# Create random Q, K, V
Q = torch.randn(batch_size, seq_len, d_k)
K = torch.randn(batch_size, seq_len, d_k)
V = torch.randn(batch_size, seq_len, d_v)

# Compute attention
output, attn_weights = scaled_dot_product_attention(Q, K, V)

print(f"\n输入形状:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")

print(f"\n输出形状:")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {attn_weights.shape}")

print(f"\n验证:")
print(f"  注意力权重每行和: {attn_weights[0].sum(dim=-1)}")
print(f"  (应该都接近 1.0)")

# Visualize attention pattern
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(attn_weights[0].detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Pattern (Sample 1)')

plt.subplot(1, 2, 2)
plt.imshow(attn_weights[1].detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Attention Pattern (Sample 2)')

plt.tight_layout()
plt.show()

print("\n✓ 缩放点积注意力实现成功！")

## 3. 自注意力原理 (Self-Attention Theory)

### 3.1 核心概念：Query, Key, Value

自注意力的核心是三个概念：

**Query (查询)**：
- "我想要什么信息？"
- 当前位置的查询向量

**Key (键)**：
- "我有什么信息？"
- 每个位置的键向量

**Value (值)**：
- "我提供的具体信息"
- 每个位置的值向量

**类比**：搜索引擎
- Query：你的搜索词
- Key：网页的标题/关键词
- Value：网页的实际内容

### 3.2 缩放点积注意力

**数学公式**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**计算步骤**：

1. **计算相似度**：$\text{scores} = QK^T$
   - 点积衡量 Query 和 Key 的相似度
   - 形状：$(n, d_k) \times (d_k, n) = (n, n)$

2. **缩放**：$\text{scores} = \frac{QK^T}{\sqrt{d_k}}$
   - 除以 $\sqrt{d_k}$ 防止点积过大
   - 保持梯度稳定

3. **归一化**：$\text{weights} = \text{softmax}(\text{scores})$
   - 转换为概率分布
   - 每行和为 1

4. **加权求和**：$\text{output} = \text{weights} \cdot V$
   - 根据注意力权重聚合信息
   - 形状：$(n, n) \times (n, d_v) = (n, d_v)$

### 3.3 为什么需要缩放？

当 $d_k$ 很大时，点积的方差会很大：

$$\text{Var}(QK^T) = d_k$$

导致 softmax 进入饱和区，梯度很小。缩放后：

$$\text{Var}\left(\frac{QK^T}{\sqrt{d_k}}\right) = 1$$

保持梯度稳定，训练更容易。

## 2. 注意力机制回顾 (Attention Recap)

### 2.1 Seq2Seq 中的注意力

在 Seq2Seq 模型中，注意力机制解决了固定长度上下文向量的瓶颈：

**传统 Seq2Seq**：
```
编码器 → 固定向量 → 解码器
```
问题：长序列信息丢失

**带注意力的 Seq2Seq**：
```
编码器 → 所有隐藏状态 → 注意力 → 解码器
```
优势：解码器可以关注源序列的不同部分

### 2.2 传统注意力 vs 自注意力

**传统注意力（Encoder-Decoder Attention）**：
- Query：来自解码器
- Key & Value：来自编码器
- 用途：对齐源序列和目标序列

**自注意力（Self-Attention）**：
- Query、Key、Value：都来自同一序列
- 用途：捕获序列内部的依赖关系
- 优势：可以并行计算

### 2.3 为什么需要自注意力？

**问题**：如何让模型理解句子内部的关系？

例句："The animal didn't cross the street because **it** was too tired."

- "it" 指代什么？
- 需要关注 "animal" 而非 "street"
- 自注意力可以学习这种关系

# Module 3.1: 自注意力机制 (Self-Attention)

## 1. 本章概览 (Overview)

### 📚 学习目标

1. **注意力机制回顾**：理解 Seq2Seq 中的注意力
2. **自注意力原理**：理解 Query、Key、Value 概念
3. **缩放点积注意力**：实现核心注意力计算
4. **多头注意力**：理解并实现多头机制
5. **注意力可视化**：直观理解注意力模式

### 🎯 核心问题

- 什么是自注意力？与传统注意力有何不同？
- Query、Key、Value 分别代表什么？
- 为什么需要缩放？
- 多头注意力的优势是什么？

### 🗺️ 注意力演进

```
传统注意力 (Seq2Seq)
  ↓
自注意力 (Self-Attention)
  ↓
多头注意力 (Multi-Head Attention)
  ↓
Transformer
```

### ⏱️ 预计学习时间：2-3小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")